In [0]:
%sql
  CREATE WIDGET TEXT catalog DEFAULT 'bootcamp';
  USE CATALOG IDENTIFIER(:catalog);

# Phase 3 - Silver cleansing
Builds the cleansed, typed and de-duplicated silver tables from bronze, and routes invalid rows to `etl_rejects`.
- **Master data** (customers, products, stores): trim text, `initcap(trim(city))`, blank → NULL, real types (DATE, DECIMAL(10,2) for money, INT, BOOLEAN), customers de-duplicated with `row_number()` (508 → 500).
- **Transactions** (orders, order_items): each source row is tagged in a validation view with a `reject_reason` (NULL = valid). Valid rows go to the silver table, invalid rows to `etl_rejects`. Because both come from the same view, every bronze row ends up in exactly one place.
- `order_ts` is parsed with two formats (`yyyy-MM-dd HH:mm:ss`, then `dd/MM/yyyy HH:mm`) instead of rejecting the 76 differently formatted orders.
- Order lines are checked against **silver** orders, so a line can never survive without a valid order header.
- Line amounts (`gross_amount`, `discount_amount`, `net_amount`) are calculated once here and reused in gold.
- Guest orders (blank customer) and unknown products (PX000) are **not** rejected: they are real sales and get the unknown member −1 in gold.

In [0]:
CREATE TABLE IF NOT EXISTS silver_efkleia_urbanmart.customers (
  customer_id   STRING    NOT NULL COMMENT 'Business key (unique after de-duplication).',
  first_name    STRING             COMMENT 'First name (trimmed).',
  last_name     STRING             COMMENT 'Last name (trimmed).',
  email         STRING             COMMENT 'Email (trimmed); NULL when blank in source.',
  gender        STRING             COMMENT 'Gender (F/M).',
  birth_date    DATE               COMMENT 'Birth date.',
  city          STRING             COMMENT 'City, standardised with initcap(trim()).',
  region        STRING             COMMENT 'Region (trimmed).',
  segment       STRING             COMMENT 'Customer segment (Retail / Loyalty Silver / Loyalty Gold).',
  signup_date   DATE               COMMENT 'Signup date.',
  _source_file  STRING    NOT NULL COMMENT 'Lineage: source file path (from bronze).',
  _ingested_at  TIMESTAMP NOT NULL COMMENT 'Lineage: bronze ingestion timestamp.',
  _silver_loaded_at TIMESTAMP NOT NULL COMMENT 'Silver load timestamp.'
)
COMMENT 'Silver: cleansed, typed, de-duplicated customers (one row per customer_id).';

INSERT OVERWRITE silver_efkleia_urbanmart.customers
  (customer_id, first_name, last_name, email, gender, birth_date, city, region, segment, signup_date,
   _source_file, _ingested_at, _silver_loaded_at)
SELECT
  customer_id, first_name, last_name, email, gender, birth_date, city, region, segment, signup_date,
  _source_file, _ingested_at, current_timestamp()
FROM (
  SELECT
    trim(customer_id)                         AS customer_id,
    trim(first_name)                          AS first_name,
    trim(last_name)                           AS last_name,
    nullif(trim(email), '')                   AS email,
    upper(trim(gender))                       AS gender,
    try_to_date(trim(birth_date),  'yyyy-MM-dd') AS birth_date,
    initcap(trim(city))                       AS city,
    trim(region)                              AS region,
    trim(segment)                             AS segment,
    try_to_date(trim(signup_date), 'yyyy-MM-dd') AS signup_date,
    _source_file,
    _ingested_at,
    row_number() OVER (PARTITION BY trim(customer_id)
                       ORDER BY _source_row_hash)  AS rn
  FROM bronze_efkleia_urbanmart.customers
  WHERE nullif(trim(customer_id), '') IS NOT NULL
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows
500,500


In [0]:
SELECT COUNT(*)                    AS total_rows,         -- expect 500
       COUNT(DISTINCT customer_id) AS distinct_customers, -- expect 500
       COUNT_IF(email IS NULL)     AS null_emails,        -- expect 15
       COUNT(DISTINCT city)        AS distinct_cities     -- expect 7
FROM silver_efkleia_urbanmart.customers;

total_rows,distinct_customers,null_emails,distinct_cities
500,500,15,7


In [0]:
CREATE TABLE IF NOT EXISTS silver_efkleia_urbanmart.products (
  product_id    STRING        NOT NULL COMMENT 'Business key.',
  product_name  STRING                 COMMENT 'Product name (trimmed).',
  category      STRING                 COMMENT 'Category; NULL when blank in source.',
  subcategory   STRING                 COMMENT 'Subcategory; NULL when blank in source.',
  brand         STRING                 COMMENT 'Brand; NULL when blank in source.',
  unit_cost     DECIMAL(10,2)          COMMENT 'Purchase cost per unit.',
  list_price    DECIMAL(10,2)          COMMENT 'Selling list price per unit.',
  is_active     BOOLEAN                COMMENT 'TRUE when source flag = Y.',
  _source_file  STRING        NOT NULL COMMENT 'Lineage: source file path (from bronze).',
  _ingested_at  TIMESTAMP     NOT NULL COMMENT 'Lineage: bronze ingestion timestamp.',
  _silver_loaded_at TIMESTAMP NOT NULL COMMENT 'Silver load timestamp.'
)
COMMENT 'Silver: cleansed, typed products. Includes placeholder P9999 (excluded later in gold dim_product).';

INSERT OVERWRITE silver_efkleia_urbanmart.products
  (product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active,
   _source_file, _ingested_at, _silver_loaded_at)
SELECT
  trim(product_id),
  trim(product_name),
  nullif(trim(category),    ''),
  nullif(trim(subcategory), ''),
  nullif(trim(brand),       ''),
  try_cast(trim(unit_cost)  AS DECIMAL(10,2)),
  try_cast(trim(list_price) AS DECIMAL(10,2)),
  upper(trim(is_active)) = 'Y',
  _source_file,
  _ingested_at,
  current_timestamp()
FROM bronze_efkleia_urbanmart.products
WHERE nullif(trim(product_id), '') IS NOT NULL;

num_affected_rows,num_inserted_rows
76,76


In [0]:
CREATE TABLE IF NOT EXISTS silver_efkleia_urbanmart.stores (
  store_id      STRING    NOT NULL COMMENT 'Business key.',
  store_name    STRING             COMMENT 'Store name (trimmed).',
  store_type    STRING             COMMENT 'High Street / Mall / Outlet / Online.',
  city          STRING             COMMENT 'City (trimmed).',
  region        STRING             COMMENT 'Region (trimmed).',
  open_date     DATE               COMMENT 'Store opening date.',
  size_sqm      INT                COMMENT 'Floor area in m2; 0 for the online store.',
  _source_file  STRING    NOT NULL COMMENT 'Lineage: source file path (from bronze).',
  _ingested_at  TIMESTAMP NOT NULL COMMENT 'Lineage: bronze ingestion timestamp.',
  _silver_loaded_at TIMESTAMP NOT NULL COMMENT 'Silver load timestamp.'
)
COMMENT 'Silver: cleansed, typed stores (12 physical + 1 online).';

INSERT OVERWRITE silver_efkleia_urbanmart.stores
  (store_id, store_name, store_type, city, region, open_date, size_sqm,
   _source_file, _ingested_at, _silver_loaded_at)
SELECT
  trim(store_id),
  trim(store_name),
  trim(store_type),
  initcap(trim(city)),
  trim(region),
  try_to_date(trim(open_date), 'yyyy-MM-dd'),
  try_cast(trim(size_sqm) AS INT),
  _source_file,
  _ingested_at,
  current_timestamp()
FROM bronze_efkleia_urbanmart.stores
WHERE nullif(trim(store_id), '') IS NOT NULL;

num_affected_rows,num_inserted_rows
13,13


In [0]:
SELECT 'products' AS tbl,
       COUNT(*)                     AS total_rows,      -- expect 76
       COUNT_IF(category IS NULL)   AS null_category,   -- expect 1 (P9999)
       COUNT_IF(unit_cost IS NULL)  AS null_cost,       -- expect 0
       COUNT_IF(is_active)          AS active           -- expect 55
FROM silver_efkleia_urbanmart.products
UNION ALL
SELECT 'stores',
       COUNT(*),                                         -- expect 13
       COUNT_IF(open_date IS NULL),                      -- expect 0
       COUNT_IF(size_sqm IS NULL),                       -- expect 0
       COUNT_IF(store_type = 'Online')                   -- expect 1
FROM silver_efkleia_urbanmart.stores;

tbl,total_rows,null_category,null_cost,active
products,76,1,0,55
stores,13,0,0,1


In [0]:
DESCRIBE silver_efkleia_urbanmart.products;

col_name,data_type,comment
product_id,string,Business key.
product_name,string,Product name (trimmed).
category,string,Category; NULL when blank in source.
subcategory,string,Subcategory; NULL when blank in source.
brand,string,Brand; NULL when blank in source.
unit_cost,"decimal(10,2)",Purchase cost per unit.
list_price,"decimal(10,2)",Selling list price per unit.
is_active,boolean,TRUE when source flag = Y.
_source_file,string,Lineage: source file path (from bronze).
_ingested_at,timestamp,Lineage: bronze ingestion timestamp.


❗ **Stretch (SCD2):** cleansed customer change events, the source for `dim_customer_scd2` in gold. Not used by the base star schema.

In [0]:
CREATE TABLE IF NOT EXISTS silver_efkleia_urbanmart.customer_changes (
  customer_id       STRING    NOT NULL COMMENT 'Customer business key.',
  new_city          STRING             COMMENT 'City from the effective date on.',
  new_region        STRING             COMMENT 'Region from the effective date on.',
  new_segment       STRING             COMMENT 'Segment from the effective date on.',
  effective_date    DATE      NOT NULL COMMENT 'Date the change takes effect.',
  _source_file      STRING    NOT NULL COMMENT 'Lineage: source file path (from bronze).',
  _ingested_at      TIMESTAMP NOT NULL COMMENT 'Lineage: bronze ingestion timestamp.',
  _silver_loaded_at TIMESTAMP NOT NULL COMMENT 'Silver load timestamp.'
)
COMMENT 'Silver: cleansed customer change events, source for dim_customer_scd2.';

INSERT OVERWRITE silver_efkleia_urbanmart.customer_changes
  (customer_id, new_city, new_region, new_segment, effective_date,
   _source_file, _ingested_at, _silver_loaded_at)
SELECT
  trim(customer_id),
  initcap(trim(new_city)),
  trim(new_region),
  trim(new_segment),
  try_to_date(trim(effective_date), 'yyyy-MM-dd'),
  _source_file,
  _ingested_at,
  current_timestamp()
FROM bronze_efkleia_urbanmart.customer_changes
WHERE nullif(trim(customer_id), '') IS NOT NULL
  AND try_to_date(trim(effective_date), 'yyyy-MM-dd') IS NOT NULL;

num_affected_rows,num_inserted_rows
25,25


In [0]:
CREATE TABLE IF NOT EXISTS silver_efkleia_urbanmart.etl_rejects (
  source_table   STRING    NOT NULL COMMENT 'Bronze table the rejected row came from (orders / order_items).',
  source_key     STRING             COMMENT 'Business key of the rejected row (order_id / order_line_id).',
  reject_reason  STRING    NOT NULL COMMENT 'Reason code, e.g. NEGATIVE_QUANTITY, ORPHAN_ORDER.',
  raw_record     STRING    NOT NULL COMMENT 'Full raw bronze row as JSON, for investigation.',
  _source_file   STRING    NOT NULL COMMENT 'Lineage: source file path (from bronze).',
  _ingested_at   TIMESTAMP NOT NULL COMMENT 'Lineage: bronze ingestion timestamp.',
  _rejected_at   TIMESTAMP NOT NULL COMMENT 'Timestamp the row was rejected.'
)
COMMENT 'Silver: rows that failed data-quality rules and were NOT loaded to silver/gold.';

In [0]:
CREATE OR REPLACE TEMP VIEW v_orders_checked AS
SELECT
  order_id, order_ts, order_date, ts_reformatted, customer_id, store_id,
  payment_method, order_status, raw_record, _source_file, _ingested_at,
  CASE
    WHEN order_id IS NULL                      THEN 'MISSING_ORDER_ID'
    WHEN rn > 1                                THEN 'DUPLICATE_ORDER'
    WHEN order_ts IS NULL                      THEN 'INVALID_ORDER_TS'
    WHEN store_id IS NULL OR NOT store_exists  THEN 'UNKNOWN_STORE'
  END AS reject_reason
FROM (
  SELECT
    nullif(trim(o.order_id), '')                                        AS order_id,
    coalesce(try_to_timestamp(trim(o.order_ts), 'yyyy-MM-dd HH:mm:ss'),
             try_to_timestamp(trim(o.order_ts), 'dd/MM/yyyy HH:mm'))    AS order_ts,
    to_date(coalesce(try_to_timestamp(trim(o.order_ts), 'yyyy-MM-dd HH:mm:ss'),
                     try_to_timestamp(trim(o.order_ts), 'dd/MM/yyyy HH:mm'))) AS order_date,
    try_to_timestamp(trim(o.order_ts), 'yyyy-MM-dd HH:mm:ss') IS NULL   AS ts_reformatted,
    nullif(trim(o.customer_id), '')                                     AS customer_id,
    nullif(trim(o.store_id), '')                                        AS store_id,
    trim(o.payment_method)                                              AS payment_method,
    trim(o.order_status)                                                AS order_status,
    s.store_id IS NOT NULL                                              AS store_exists,
    to_json(named_struct('order_id', o.order_id, 'order_ts', o.order_ts,
                         'customer_id', o.customer_id, 'store_id', o.store_id,
                         'payment_method', o.payment_method, 'order_status', o.order_status)) AS raw_record,
    o._source_file,
    o._ingested_at,
    row_number() OVER (PARTITION BY trim(o.order_id) ORDER BY o._source_row_hash) AS rn
  FROM bronze_efkleia_urbanmart.orders o
  LEFT JOIN silver_efkleia_urbanmart.stores s
    ON trim(o.store_id) = s.store_id
);

In [0]:
CREATE TABLE IF NOT EXISTS silver_efkleia_urbanmart.orders (
  order_id        STRING    NOT NULL COMMENT 'Business key.',
  order_ts        TIMESTAMP NOT NULL COMMENT 'Order timestamp (parsed from 2 source formats).',
  order_date      DATE      NOT NULL COMMENT 'Order date (date part of order_ts).',
  ts_reformatted  BOOLEAN   NOT NULL COMMENT 'TRUE when source used dd/MM/yyyy HH:mm (time is a 12:00 placeholder).',
  customer_id     STRING             COMMENT 'Customer business key; NULL = guest order.',
  store_id        STRING    NOT NULL COMMENT 'Store business key.',
  payment_method  STRING             COMMENT 'Card / Cash / Mobile Wallet.',
  order_status    STRING             COMMENT 'Completed / Returned / Cancelled.',
  _source_file    STRING    NOT NULL COMMENT 'Lineage: source file path (from bronze).',
  _ingested_at    TIMESTAMP NOT NULL COMMENT 'Lineage: bronze ingestion timestamp.',
  _silver_loaded_at TIMESTAMP NOT NULL COMMENT 'Silver load timestamp.'
)
COMMENT 'Silver: cleansed, typed order headers. Invalid orders are in etl_rejects.';

INSERT OVERWRITE silver_efkleia_urbanmart.orders
  (order_id, order_ts, order_date, ts_reformatted, customer_id, store_id, payment_method, order_status,
   _source_file, _ingested_at, _silver_loaded_at)
SELECT
  order_id, order_ts, order_date, ts_reformatted, customer_id, store_id, payment_method, order_status,
  _source_file, _ingested_at, current_timestamp()
FROM v_orders_checked
WHERE reject_reason IS NULL;

num_affected_rows,num_inserted_rows
8000,8000


In [0]:
CREATE OR REPLACE TEMP VIEW v_order_items_checked AS
SELECT
  order_line_id, order_id, product_id, quantity, unit_price, discount_pct,
  raw_record, _source_file, _ingested_at,
  CASE
    WHEN order_line_id IS NULL                        THEN 'MISSING_LINE_ID'
    WHEN rn > 1                                       THEN 'DUPLICATE_LINE'
    WHEN NOT order_exists                             THEN 'ORPHAN_ORDER'
    WHEN quantity IS NULL                             THEN 'INVALID_QUANTITY'
    WHEN quantity < 0                                 THEN 'NEGATIVE_QUANTITY'
    WHEN quantity = 0                                 THEN 'ZERO_QUANTITY'
    WHEN unit_price IS NULL OR unit_price < 0         THEN 'INVALID_UNIT_PRICE'
    WHEN discount_pct IS NULL
      OR discount_pct NOT BETWEEN 0 AND 1             THEN 'INVALID_DISCOUNT'
  END AS reject_reason
FROM (
  SELECT
    nullif(trim(i.order_line_id), '')                    AS order_line_id,
    nullif(trim(i.order_id), '')                         AS order_id,
    nullif(trim(i.product_id), '')                       AS product_id,
    try_cast(trim(i.quantity)     AS INT)                AS quantity,
    try_cast(trim(i.unit_price)   AS DECIMAL(10,2))      AS unit_price,
    try_cast(trim(i.discount_pct) AS DECIMAL(5,4))       AS discount_pct,
    o.order_id IS NOT NULL                               AS order_exists,
    to_json(named_struct('order_line_id', i.order_line_id, 'order_id', i.order_id,
                         'product_id', i.product_id, 'quantity', i.quantity,
                         'unit_price', i.unit_price, 'discount_pct', i.discount_pct)) AS raw_record,
    i._source_file,
    i._ingested_at,
    row_number() OVER (PARTITION BY trim(i.order_line_id) ORDER BY i._source_row_hash) AS rn
  FROM bronze_efkleia_urbanmart.order_items i
  LEFT JOIN silver_efkleia_urbanmart.orders o
    ON trim(i.order_id) = o.order_id
);

In [0]:
CREATE TABLE IF NOT EXISTS silver_efkleia_urbanmart.order_items (
  order_line_id    STRING        NOT NULL COMMENT 'Business key.',
  order_id         STRING        NOT NULL COMMENT 'FK to silver.orders.',
  product_id       STRING                 COMMENT 'Product business key (may not exist in products, e.g. PX000).',
  quantity         INT           NOT NULL COMMENT 'Units sold (> 0).',
  unit_price       DECIMAL(10,2) NOT NULL COMMENT 'Selling price per unit.',
  discount_pct     DECIMAL(5,4)  NOT NULL COMMENT 'Discount fraction 0-1.',
  gross_amount     DECIMAL(12,2) NOT NULL COMMENT 'quantity x unit_price.',
  discount_amount  DECIMAL(12,2) NOT NULL COMMENT 'gross_amount x discount_pct, rounded to 2 decimals.',
  net_amount       DECIMAL(12,2) NOT NULL COMMENT 'gross_amount - discount_amount.',
  _source_file     STRING        NOT NULL COMMENT 'Lineage: source file path (from bronze).',
  _ingested_at     TIMESTAMP     NOT NULL COMMENT 'Lineage: bronze ingestion timestamp.',
  _silver_loaded_at TIMESTAMP    NOT NULL COMMENT 'Silver load timestamp.'
)
COMMENT 'Silver: cleansed, typed order lines with line amounts. Invalid lines are in etl_rejects.';

INSERT OVERWRITE silver_efkleia_urbanmart.order_items
  (order_line_id, order_id, product_id, quantity, unit_price, discount_pct,
   gross_amount, discount_amount, net_amount,
   _source_file, _ingested_at, _silver_loaded_at)
SELECT
  order_line_id, order_id, product_id, quantity, unit_price, discount_pct,
  CAST(quantity * unit_price AS DECIMAL(12,2))                              AS gross_amount,
  CAST(round(quantity * unit_price * discount_pct, 2) AS DECIMAL(12,2))     AS discount_amount,
  CAST(quantity * unit_price
       - round(quantity * unit_price * discount_pct, 2) AS DECIMAL(12,2))   AS net_amount,
  _source_file, _ingested_at, current_timestamp()
FROM v_order_items_checked
WHERE reject_reason IS NULL;

num_affected_rows,num_inserted_rows
23997,23997


In [0]:
INSERT OVERWRITE silver_efkleia_urbanmart.etl_rejects
  (source_table, source_key, reject_reason, raw_record, _source_file, _ingested_at, _rejected_at)
SELECT 'orders', order_id, reject_reason, raw_record, _source_file, _ingested_at, current_timestamp()
FROM v_orders_checked
WHERE reject_reason IS NOT NULL
UNION ALL
SELECT 'order_items', order_line_id, reject_reason, raw_record, _source_file, _ingested_at, current_timestamp()
FROM v_order_items_checked
WHERE reject_reason IS NOT NULL;

num_affected_rows,num_inserted_rows
114,114


In [0]:
SELECT 'silver.orders' AS check_name, COUNT(*) AS n FROM silver_efkleia_urbanmart.orders          -- expect 8000
UNION ALL SELECT 'orders reformatted ts', COUNT_IF(ts_reformatted) FROM silver_efkleia_urbanmart.orders  -- expect 76
UNION ALL SELECT 'guest orders', COUNT_IF(customer_id IS NULL) FROM silver_efkleia_urbanmart.orders      -- expect 312
UNION ALL SELECT 'silver.order_items', COUNT(*) FROM silver_efkleia_urbanmart.order_items          -- expect 23997
UNION ALL SELECT 'rejects total', COUNT(*) FROM silver_efkleia_urbanmart.etl_rejects               -- expect 114
UNION ALL SELECT 'bronze lines', COUNT(*) FROM bronze_efkleia_urbanmart.order_items;               -- expect 24111

check_name,n
orders reformatted ts,76
guest orders,312
silver.orders,8000
silver.order_items,23997
rejects total,114
bronze lines,24111


**Result:** customers 500 (15 NULL emails, 7 cities) · products 76 · stores 13 · customer_changes 25 · orders 8,000 (76 reformatted timestamps, 312 guest orders) · order_items 23,997.
`etl_rejects`: 114 rows = 113 NEGATIVE_QUANTITY + 1 ORPHAN_ORDER.
Reconciliation: 24,111 bronze lines − 114 rejected = 23,997 silver lines.

In [0]:
SELECT source_table, reject_reason, COUNT(*) AS n
FROM silver_efkleia_urbanmart.etl_rejects
GROUP BY source_table, reject_reason
ORDER BY n DESC;
-- expect: order_items | NEGATIVE_QUANTITY | 113
--         order_items | ORPHAN_ORDER      | 1

source_table,reject_reason,n
order_items,NEGATIVE_QUANTITY,113
order_items,ORPHAN_ORDER,1
